# Análise de Redução de Dimensionalidade no LinearSVC: Seleção e Projeção (PCA / SVD)

**Pergunta central:** Técnicas de redução de dimensionalidade conseguem melhorar o F1 da validação cruzada do `LinearSVC` ou mitigar o gap de generalização? O que acontece quando reduzimos o espaço para o regime onde o número de dimensões é muito menor que o número de instâncias ($d \ll N$, com $N = 5.760$ amostras)?

---

### 1. PCA em Matrizes Esparsas de Texto: TruncatedSVD (LSA)
Em Processamento de Linguagem Natural com representações TF-IDF:
- A matriz termo-documento possui **~114.300 colunas**, mas é **~99.5% esparsa** (a imensa maioria das entradas é zero).
- O algoritmo clássico de **PCA** (`sklearn.decomposition.PCA`) exige centralizar os dados subtraindo a média de cada coluna ($\mathbf{X} - \boldsymbol{\mu}$). Subtrair a média transforma todos os zeros em números negativos não nulos, convertendo a matriz esparsa em uma matriz 100% densa. Armazenar $5.760 \times 114.300$ números float64 densos exigiria **~5,3 GB de memória RAM contínua**, causando erro imediato de *Out of Memory* (o próprio scikit-learn rejeita matrizes esparsas no `PCA` com `TypeError`).
- A solução padrão na literatura é o **TruncatedSVD** (conhecido como **LSA - Latent Semantic Analysis**): ele calcula a decomposição em valores singulares diretamente sobre a matriz esparsa, sem centralização, preservando a eficiência de memória e capturando os mesmos eixos principais de variação semântica.
- Combinado com `Normalizer(norm="l2")`, projeta os documentos na hipersfera unitária, preservando a similaridade de cosseno.

---

### 2. Três Abordagens Comparadas
Comparamos a representação completa `word+char` (~114k features) contra:
1. **TruncatedSVD (PCA Esparso):** Projeção em $n$ componentes densos semânticos latentes, cobrindo desde **baixíssimas dimensionalidades** ($n \in [2, 3, 5, 10, 20, 50, 100, 200, 300]$) até escalas intermediárias.
2. **Seleção Univariada Supervisionada ($\chi^2$ via `SelectKBest`):** Seleciona as $k$ features com maior dependência estatística com a classe ($k \in [500, 2000, 5000, 10000, 20000, 30000, 50000, 80000]$).
3. **Seleção Baseada em Modelo ($L_1$ com `SelectFromModel`):** LinearSVC com penalização Lasso ($L_1$, `dual=False`) induzindo esparsidade ($C \in [0.1, 0.3, 1, 3]$).

> **Garantia Metodológica:** Toda a redução é ajustada estritamente dentro do `Pipeline` no treino de cada fold, garantindo ausência total de vazamento (*data leakage*) para a validação.

In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, SelectFromModel, chi2
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_validate

# Carregamento dos dados de treino e transformers
X_tr, X_te, y_tr, y_te = joblib.load("../../data/dados_preparados.pkl")
configs = joblib.load("../../data/transformers.pkl")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def svc():
    return LinearSVC(class_weight="balanced", random_state=42, max_iter=5000)

print(f"Total de instâncias de treino (N): {len(X_tr):,}")
print("Configuração base: word+char (~114.300 features)")

## Visualização Preliminar: Projeção 2D via PCA / TruncatedSVD e Variância Explicada

Antes de rodar a validação cruzada, analisamos a variância explicada das primeiras componentes e projetamos o conjunto de treino em 2 dimensões ($PC_1 \times PC_2$) para observar se as notícias falsas e verdadeiras são separáveis visualmente no espaço de baixíssima dimensão.

In [ ]:
prep = ColumnTransformer(configs["word+char"])
X_tfidf = prep.fit_transform(X_tr)

svd_vis = TruncatedSVD(n_components=10, random_state=42)
X_2d = svd_vis.fit_transform(X_tfidf)

print("Variância explicada acumulada pelas primeiras 10 componentes:")
for i, v in enumerate(np.cumsum(svd_vis.explained_variance_ratio_)):
    print(f"  PC {i+1:>2}: {v*100:.2f}% acumulado (individual: {svd_vis.explained_variance_ratio_[i]*100:.2f}%)")

# Scatter plot 2D
fig, ax = plt.subplots(figsize=(8, 5.5))
cores = {0: "#2a78d6", 1: "#e63946"}
rotulos = {0: "Notícia Real", 1: "Fake News"}

for classe in [0, 1]:
    idx = (y_tr.values == classe)
    ax.scatter(X_2d[idx, 0], X_2d[idx, 1], c=cores[classe], label=rotulos[classe], alpha=0.35, s=16)

ax.set_title("Projeção 2D do Corpus Fake.br via TruncatedSVD (PCA Esparso)", fontsize=12)
ax.set_xlabel(f"PC 1 ({svd_vis.explained_variance_ratio_[0]*100:.2f}% da variância)", fontsize=10)
ax.set_ylabel(f"PC 2 ({svd_vis.explained_variance_ratio_[1]*100:.2f}% da variância)", fontsize=10)
ax.grid(True, alpha=0.3)
ax.legend(markerscale=2)
plt.tight_layout()
plt.show()

## Execução dos Cenários de Redução de Dimensionalidade

Testamos o `LinearSVC` em 5 folds com:
- Baseline completa (~114k features)
- **PCA / TruncatedSVD com baixíssimas dimensionalidades:** $d \in [2, 3, 5, 10, 20, 50, 100, 200, 300]$
- **$\chi^2$ (SelectKBest):** $k \in [500, 2000, 5000, 10000, 20000, 30000, 50000, 80000]$
- **$L_1$ Penalty (SelectFromModel):** $C \in [0.1, 0.3, 1, 3]$

Os resultados consolidados são salvos em `../resultados/resultados_reducao_dim.csv`.

In [ ]:
cenarios = [("base", None, 114301, [])]
# Baixíssimas e médias dimensionalidades de PCA / TruncatedSVD
cenarios += [("svd (pca)", n, n, [("svd", TruncatedSVD(n_components=n, random_state=42)), ("norm", Normalizer())])
             for n in [2, 3, 5, 10, 20, 50, 100, 200, 300]]
# Seleção supervisionada chi2
cenarios += [("chi2", k, k, [("sel", SelectKBest(chi2, k=k))]) for k in [500, 2000, 5000, 10000, 20000, 30000, 50000, 80000]]
# Seleção baseada em modelo L1
cenarios += [("l1", C, None, [("sel", SelectFromModel(LinearSVC(penalty="l1", dual=False, C=C, class_weight="balanced")))])
             for C in [0.1, 0.3, 1, 3]]

csv_reducao = "../resultados/resultados_reducao_dim.csv"

if os.path.exists(csv_reducao):
    print(f"Carregando resultados consolidados de {csv_reducao}...")
    res = pd.read_csv(csv_reducao)
else:
    print("Executando validação cruzada para todos os cenários...")
    linhas = []
    for metodo, param, n_feats, passos in cenarios:
        pipe = Pipeline([("prep", ColumnTransformer(configs["word+char"]))] + passos + [("clf", svc())])
        inicio = time.time()
        r = cross_validate(pipe, X_tr, y_tr, cv=cv, scoring="f1_macro", return_train_score=True,
                           return_estimator=(metodo == "l1"), n_jobs=2)
        if metodo == "l1":
            n_feats = int(np.mean([e.named_steps["sel"].get_support().sum() for e in r["estimator"]]))
        linhas.append({
            "metodo": metodo,
            "param": param,
            "n_feats": n_feats,
            "f1_treino": r["train_score"].mean(),
            "f1_cv": r["test_score"].mean(),
            "f1_cv_std": r["test_score"].std(),
            "gap": r["train_score"].mean() - r["test_score"].mean(),
            "tempo_s": time.time() - inicio,
            **{f"fold{i+1}": s for i, s in enumerate(r["test_score"])}
        })
        l = linhas[-1]
        print(f"{metodo:>9} {str(param):>6} | feats={n_feats:>7,} | treino={l['f1_treino']:.4f} cv={l['f1_cv']:.4f} ± {l['f1_cv_std']:.4f} | {l['tempo_s']:.0f}s", flush=True)

    res = pd.DataFrame(linhas)
    folds = [f"fold{i}" for i in range(1, 6)]
    base = res.loc[res.metodo == "base", folds].values[0]
    res["delta_cv"] = [(row - base).mean() for row in res[folds].values]
    res["delta_std"] = [(row - base).std() for row in res[folds].values]
    res["folds_melhores"] = [int((row > base).sum()) for row in res[folds].values]
    res.to_csv(csv_reducao, index=False)

## Tabela Comparativa de Resultados

In [ ]:
colunas_exibir = ["metodo", "param", "n_feats", "f1_treino", "f1_cv", "delta_cv", "folds_melhores", "gap"]
print("=== Resultados de Redução de Dimensionalidade (Ordenados por F1 CV) ===")
display(res[colunas_exibir].sort_values("f1_cv", ascending=False).round(4).reset_index(drop=True))

## Visualizações: F1 CV vs Dimensionalidade (Escala Log)

Abaixo comparamos o comportamento da projeção não supervisionada (**TruncatedSVD / PCA**) versus a seleção supervisionada (**$\chi^2$**) em função do número de dimensões $d$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Curva F1 CV vs Dimensão para SVD (PCA) e Chi2
svd_df = res[res.metodo.str.contains("svd")].sort_values("n_feats")
chi2_df = res[res.metodo == "chi2"].sort_values("n_feats")
base_f1 = res[res.metodo == "base"]["f1_cv"].iloc[0]

axes[0].plot(svd_df["n_feats"], svd_df["f1_cv"], marker="s", lw=2, color="#eb6834", label="TruncatedSVD (PCA denso)")
axes[0].plot(chi2_df["n_feats"], chi2_df["f1_cv"], marker="o", lw=2, color="#2a78d6", label=r"$\chi^2$ (SelectKBest esparso)")
axes[0].axhline(base_f1, color="#222222", linestyle="--", lw=1.5, label="Base (~114k feats)")
axes[0].axvline(len(X_tr), color="gray", linestyle=":", label=f"N linhas treino ({len(X_tr):,})")

axes[0].set_xscale("log")
axes[0].set_title("F1 CV vs Número de Dimensões (Escala Log)", fontsize=12)
axes[0].set_xlabel("Número de Dimensões / Features (log)", fontsize=10)
axes[0].set_ylabel("F1 Macro (CV)", fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=9)

# 2. Curva do Gap de Generalização (F1 Treino - F1 CV)
axes[1].plot(svd_df["n_feats"], svd_df["gap"], marker="s", lw=2, color="#eb6834", label="Gap SVD (PCA)")
axes[1].plot(chi2_df["n_feats"], chi2_df["gap"], marker="o", lw=2, color="#2a78d6", label=r"Gap $\chi^2$")
axes[1].axhline(res[res.metodo == "base"]["gap"].iloc[0], color="#222222", linestyle="--", label="Gap Base")
axes[1].axvline(len(X_tr), color="gray", linestyle=":", label=f"N linhas ({len(X_tr):,})")

axes[1].set_xscale("log")
axes[1].set_title("Gap de Generalização ($F1_{treino} - F1_{CV}$)", fontsize=12)
axes[1].set_xlabel("Número de Dimensões (log)", fontsize=10)
axes[1].set_ylabel("Gap", fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Conclusões sobre a Redução de Dimensionalidade e Baixas Dimensões ($d \ll N$)

1. **Baixíssimas Dimensionalidades ($d \in [2, 5]$):**
   - Com $d=2$ e $d=3$, o F1 da validação cruzada fica em torno de **0.548 - 0.551**, mal superando o acaso (0.50).
   - O gráfico de dispersão 2D revela o motivo: os dois primeiros componentes explicam apenas **0.75% da variância total** e capturam aspectos genéricos da linguagem (tamanho do texto, palavras funcionais), misturando completamente fakes e reais.
2. **Evolução com o aumento de componentes no SVD:**
   - Com $d=10$: F1 sobe para **0.7812**.
   - Com $d=50$: F1 atinge **0.8663**.
   - Com $d=300$: F1 chega a **0.9062**.
   - Apesar da melhora consistente, o SVD ainda fica ~0.023 abaixo da baseline completa (~114k features com F1=0.9295).
3. **Comportamento do Gap (Underfitting vs Overfitting):**
   - Para $d \le 20$, o Gap é praticamente **zero** (ou levemente negativo): o modelo linear está em *underfitting* severo (alto viés), sem flexibilidade suficiente para ajustar os dados nem no treino.
   - O gap se abre conforme $d$ ultrapassa 50, refletindo a capacidade do modelo linear de traçar fronteiras de decisão mais ricas.
4. **Comparação: Projeção Densa (PCA/SVD) vs Seleção Esparsa ($\chi^2$):**
   - A seleção univariada $\chi^2$ preserva a esparsidade e a identidade dos n-gramas específicos de fake news, superando com folga o SVD com o mesmo número de features (ex.: $\chi^2$ com $k=2.000$ já atinge F1=0.8951, superando SVD com 100 componentes).
5. **Veredito:** Modelos lineares para classificação textual se beneficiam fortemente de alta dimensionalidade esparsa ($d \gg N$), conforme previsto pelo Teorema de Cover. Reduzir para baixas dimensões compacta informações sutis indispensáveis para a detecção de fake news.